# Model Retraining After Batch 02

## Objective

This notebook evaluates machine learning performance after expanding the Frankenstein therapeutic annotation dataset.

The updated dataset contains 39 manually annotated passages. Multiple classical machine learning models are tested to evaluate whether increased annotation improves therapeutic classification.

In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
from google.colab import files

uploaded = files.upload()

Saving frankenstein_training_dataset_v2.csv to frankenstein_training_dataset_v2.csv


In [3]:
df = pd.read_csv(
    "frankenstein_training_dataset_v2.csv"
)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully
Dataset shape: (39, 7)


,passage_id,text,therapeutic_label,mechanism,intensity,annotator_notes,keyword_score
0,10,This appearance excited our unqualified wonder...,0.0,NaN,0.0,The passage describes curiosity and wonder dur...,NaN
1,16,Will you smile at the enthusiasm I express con...,1.0,Narrative Reconstruction,3.0,The passage introduces Victor's suffering and ...,NaN
2,26,I feel exquisite pleasure in dwelling on the r...,1.0,Narrative Reconstruction,4.0,The passage represents autobiographical reflec...,NaN
3,67,"“She most of all,” said Ernest, “requires cons...",0.0,NaN,0.0,"The passage describes grief, guilt, and emotio...",NaN
4,127,“I knocked. ‘Who is there?’ said the old man. ...,0.0,NaN,0.0,The passage represents a compassionate social ...,NaN


## Class Distribution Analysis

The distribution of therapeutic and non-therapeutic passages is examined to evaluate dataset balance after annotation expansion.

In [4]:
df["therapeutic_label"].value_counts()

,count
therapeutic_label,
0.0,20
1.0,19


In [5]:
label_distribution = df["therapeutic_label"].value_counts(normalize=True) * 100

label_distribution

,proportion
therapeutic_label,
0.0,51.282051
1.0,48.717949


## Class Distribution Findings

After Batch 02 annotation expansion, the dataset contains 39 annotated passages with a balanced distribution between therapeutic and non-therapeutic classes.

The dataset consists of 20 non-therapeutic passages (51.28%) and 19 therapeutic passages (48.72%). This improved balance reduces class bias observed in earlier experiments and provides a stronger foundation for evaluating classification performance.

## TF-IDF Feature Extraction

Text passages are converted into numerical feature representations using TF-IDF. This allows machine learning models to identify important linguistic patterns associated with therapeutic classification.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words="english"
)

X = tfidf.fit_transform(
    df["text"]
)

y = df["therapeutic_label"]

print("Feature extraction completed")
print("Feature shape:", X.shape)

Feature extraction completed
Feature shape: (39, 1000)


## TF-IDF Feature Extraction Results

The 39 annotated Frankenstein passages were transformed into numerical representations using TF-IDF vectorization.

A feature matrix of 39 documents and 1000 linguistic features was generated. These features represent the importance of words across passages and are used as input for subsequent machine learning classification models.

## Train-Test Split

The dataset is divided into training and testing subsets while maintaining the proportion of therapeutic and non-therapeutic labels.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Training samples: 31
Testing samples: 8

Training label distribution:
therapeutic_label
0.0    16
1.0    15
Name: count, dtype: int64

Testing label distribution:
therapeutic_label
0.0    4
1.0    4
Name: count, dtype: int64


## Train-Test Split Results

The balanced dataset was divided into training and testing subsets using stratified sampling. The training set contained 31 passages, while the testing set contained 8 passages.

Stratification preserved class balance, with both training and testing sets containing comparable proportions of therapeutic and non-therapeutic passages. This improves the reliability of model evaluation.

## Logistic Regression Classification

Logistic Regression is used as the first baseline classifier to evaluate whether TF-IDF features can identify therapeutic representations in the expanded dataset.

In [8]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

log_model.fit(
    X_train,
    y_train
)

print("Logistic Regression trained successfully")

Logistic Regression trained successfully


In [9]:
log_pred = log_model.predict(X_test)

print("Predicted:", log_pred)
print("Actual:", y_test.values)

Predicted: [0. 1. 0. 1. 0. 1. 0. 1.]
Actual: [0. 0. 1. 1. 0. 1. 0. 1.]


In [10]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print(
    "Logistic Regression Accuracy:",
    accuracy_score(y_test, log_pred)
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(y_test, log_pred)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        log_pred,
        zero_division=0
    )
)

Logistic Regression Accuracy: 0.75

Confusion Matrix:
[[3 1]
 [1 3]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.75      0.75      0.75         4
         1.0       0.75      0.75      0.75         4

    accuracy                           0.75         8
   macro avg       0.75      0.75      0.75         8
weighted avg       0.75      0.75      0.75         8



## Logistic Regression Results After Batch 02 Expansion

The Logistic Regression classifier achieved an accuracy of 75% on the expanded dataset containing 39 annotated passages.

The confusion matrix showed balanced performance, with equal precision and recall (0.75) for both therapeutic and non-therapeutic classes. Unlike earlier experiments, where the model failed to identify therapeutic passages, the expanded balanced dataset enabled recognition of therapeutic representations.

These results indicate that annotation expansion significantly improves model learning and reduces class bias.

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully")

Random Forest trained successfully


In [12]:
rf_pred = rf_model.predict(X_test)

print("Predicted:", rf_pred)
print("Actual:", y_test.values)

Predicted: [0. 0. 0. 0. 1. 1. 0. 0.]
Actual: [0. 0. 1. 1. 0. 1. 0. 1.]


In [13]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print(
    "Random Forest Accuracy:",
    accuracy_score(y_test, rf_pred)
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(y_test, rf_pred)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_pred,
        zero_division=0
    )
)

Random Forest Accuracy: 0.5

Confusion Matrix:
[[3 1]
 [3 1]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.50      0.75      0.60         4
         1.0       0.50      0.25      0.33         4

    accuracy                           0.50         8
   macro avg       0.50      0.50      0.47         8
weighted avg       0.50      0.50      0.47         8



## Random Forest Results After Batch 02 Expansion

The Random Forest classifier achieved an accuracy of 50% on the expanded dataset. Although it correctly identified some non-therapeutic passages, its recall for therapeutic passages remained low (0.25).

Compared with Logistic Regression, Random Forest showed weaker performance on the current small-scale TF-IDF text classification task. This suggests that linear models may better capture the relationship between textual features and therapeutic representation in limited literary datasets.

In [14]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    random_state=42
)

svm_model.fit(
    X_train,
    y_train
)

print("SVM trained successfully")

SVM trained successfully


In [15]:
svm_pred = svm_model.predict(X_test)

print("Predicted:", svm_pred)
print("Actual:", y_test.values)

Predicted: [1. 1. 1. 1. 0. 1. 0. 1.]
Actual: [0. 0. 1. 1. 0. 1. 0. 1.]


In [16]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print(
    "SVM Accuracy:",
    accuracy_score(y_test, svm_pred)
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(y_test, svm_pred)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        svm_pred,
        zero_division=0
    )
)

SVM Accuracy: 0.75

Confusion Matrix:
[[2 2]
 [0 4]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      0.50      0.67         4
         1.0       0.67      1.00      0.80         4

    accuracy                           0.75         8
   macro avg       0.83      0.75      0.73         8
weighted avg       0.83      0.75      0.73         8



In [17]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Linear SVM"
    ],
    "Accuracy": [
        accuracy_score(y_test, log_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, svm_pred)
    ]
})

results

,Model,Accuracy
0,Logistic Regression,0.75
1,Random Forest,0.50
2,Linear SVM,0.75


In [18]:
from sklearn.metrics import precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Linear SVM"
    ],
    "Accuracy": [
        accuracy_score(y_test, log_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, svm_pred)
    ],
    "Therapeutic Precision": [
        precision_score(y_test, log_pred),
        precision_score(y_test, rf_pred),
        precision_score(y_test, svm_pred)
    ],
    "Therapeutic Recall": [
        recall_score(y_test, log_pred),
        recall_score(y_test, rf_pred),
        recall_score(y_test, svm_pred)
    ],
    "Therapeutic F1": [
        f1_score(y_test, log_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, svm_pred)
    ]
})

comparison

,Model,Accuracy,Therapeutic Precision,Therapeutic Recall,Therapeutic F1
0,Logistic Regression,0.75,0.750000,0.75,0.750000
1,Random Forest,0.50,0.500000,0.25,0.333333
2,Linear SVM,0.75,0.666667,1.00,0.800000


## Model Comparison Findings

After expanding the annotated dataset to 39 passages, classical machine learning models showed improved therapeutic classification performance compared with earlier experiments.

Logistic Regression achieved balanced performance with 75% accuracy and equal precision and recall across both classes.

Random Forest showed weaker performance, particularly in identifying therapeutic passages, suggesting limitations when applied to small, sparse TF-IDF literary datasets.

Linear SVM achieved the strongest therapeutic recall by correctly identifying all therapeutic passages in the test set. However, it produced more false positives among non-therapeutic passages.

The results indicate that dataset expansion and improved class balance significantly enhanced model learning. Linear models, particularly Logistic Regression and SVM, appear more suitable for this small-scale Digital Humanities text classification task.

In [19]:
comparison.to_csv(
    "model_comparison_batch02.csv",
    index=False
)

print("Model comparison saved")

Model comparison saved


In [20]:
prediction_results = pd.DataFrame({
    "actual": y_test.values,
    "logistic_prediction": log_pred,
    "random_forest_prediction": rf_pred,
    "svm_prediction": svm_pred
})

prediction_results.to_csv(
    "batch02_model_predictions.csv",
    index=False
)

print("Prediction results saved")

Prediction results saved


In [21]:
from google.colab import files

files.download("model_comparison_batch02.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
from google.colab import files

files.download("batch02_model_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>